# MNIST 손글씨 숫자 인식: 은닉층 1개 완전연결 신경망

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunbaklee/linreg3d/blob/main/mnist/mnist_mlp.ipynb)

| 층 | 뉴런 수 | 역할 |
|---|---|---|
| 입력층 | 784 | 펼친 픽셀값을 받음 |
| 은닉층 1개 | 128 | 입력층의 모든 뉴런과 연결되며 ReLU를 적용 |
| 출력층 | 10 | 은닉층의 모든 뉴런과 연결되며 Softmax로 숫자 0~9의 예측 확률을 출력 |

Colab에는 TensorFlow가 이미 설치되어 있으므로 위에서부터 차례로 실행하면 됩니다.

## 1. 데이터 불러오기와 원본 확인

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.utils import to_categorical

tf.keras.utils.set_random_seed(42)

(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("학습 이미지:", X_train.shape, "학습 정답:", y_train.shape)
print("테스트 이미지:", X_test.shape, "테스트 정답:", y_test.shape)

전처리 전의 `X_train[0]`을 흑백 이미지로 보고, 28×28 격자에 원본 픽셀값 784개를 표시합니다. 각 값은 0(검정)부터 255(흰색)까지의 정수입니다.

In [ ]:
img = X_train[0]

fig, ax = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={"width_ratios": [1, 2]})
ax[0].imshow(img, cmap="gray")
ax[0].set_title(f"X_train[0]  label = {y_train[0]}")
ax[0].axis("off")

ax[1].imshow(img, cmap="gray")
for r in range(28):
    for c in range(28):
        v = img[r, c]
        ax[1].text(c, r, str(v), ha="center", va="center", fontsize=5,
                   color="black" if v > 127 else "white")
ax[1].set_title("raw pixel values (0 = black, 255 = white)")
ax[1].axis("off")
plt.tight_layout()
plt.show()

## 2. 픽셀값 정규화와 배열 변환
픽셀값을 0~1 범위의 실수로 바꾸고, 이미지마다 28×28 픽셀을 한 줄(784개)로 펼칩니다.

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
print("정규화 예시: 128 ->", round(128 / 255.0, 3))

X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)
print("reshape 후:", X_train.shape, X_test.shape)

np.set_printoptions(precision=3, suppress=True, linewidth=120, threshold=1000)
print("X_train[0]의 입력값 784개:")
print(X_train[0])

정답은 원-핫 인코딩으로 바꿉니다. 정답이 5이면 인덱스 5에만 1이 들어갑니다.

In [ ]:
Y_train = to_categorical(y_train, 10)
Y_test = to_categorical(y_test, 10)
print("정답", y_train[0], "->", Y_train[0].astype(int))

## 3. 완전연결 신경망 구성

In [ ]:
model = Sequential([
    Input(shape=(784,)),               # 입력층 784
    Dense(128, activation="relu"),     # 은닉층: 입력층의 모든 뉴런과 연결
    Dense(10, activation="softmax"),   # 출력층: 은닉층의 모든 뉴런과 연결
])
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 4. 모델 학습
- `validation_split=0.2`: 학습용 60,000장 중 12,000장은 검증, 48,000장은 학습에 사용합니다.
- `batch_size=32`: 한 에폭은 48,000 ÷ 32 = **1,500단계**입니다. 진행 표시의 `1500/1500`은 그 에폭의 배치를 모두 처리했다는 뜻입니다.
- `epochs=5`: 총 7,500번 가중치를 업데이트합니다.

In [ ]:
history = model.fit(X_train, Y_train,
                    epochs=5, batch_size=32,
                    validation_split=0.2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 5. 테스트와 예측 결과 표시

In [ ]:
test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"테스트 손실: {test_loss:.4f}")
print(f"테스트 정확도: {test_acc:.4f}")

테스트 이미지 10장의 실제 정답과 예측을 함께 표시합니다. `argmax`로 확률이 가장 높은 숫자를 고릅니다. 틀린 예측은 빨간색입니다.

In [ ]:
probs = model.predict(X_test[:10], verbose=0)
pred = np.argmax(probs, axis=1)

plt.figure(figsize=(14, 2.6))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap="gray")
    ok = pred[i] == y_test[i]
    plt.title(f"true {y_test[i]} / pred {pred[i]}", color="black" if ok else "red", fontsize=10)
    plt.axis("off")
plt.show()

이미지 한 장을 골라 0~9의 예측 확률을 막대그래프로 표시합니다.

In [ ]:
idx = 0
p = model.predict(X_test[idx:idx + 1], verbose=0)[0]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [1, 2]})
ax[0].imshow(X_test[idx].reshape(28, 28), cmap="gray")
ax[0].set_title(f"X_test[{idx}]  label = {y_test[idx]}")
ax[0].axis("off")
bars = ax[1].bar(range(10), p, color="#9BB8D3")
bars[np.argmax(p)].set_color("#1D6AAE")
ax[1].set_xticks(range(10))
ax[1].set_ylim(0, 1)
ax[1].set_xlabel("digit"); ax[1].set_ylabel("probability")
ax[1].set_title(f"prediction = {np.argmax(p)}  ({p.max():.2%})")
plt.tight_layout()
plt.show()